# Fine-tuning QLoRA — Assistente Virtual Médico (Tech Challenge Fase 3)

Notebook de execução do **issue #3** (Pessoa A). Roda no **Google Colab com GPU T4**
(`Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`).

A lógica do treino **não vive aqui** — vive em `src/hospital_assistant/finetuning/train.py`.
Este notebook só orquestra: instala as dependências de GPU, prepara os dados, chama
`train()`, plota as curvas e publica o adapter. É o que atende o requisito de
"projeto modularizado em Python" do PDF sem transformar o notebook no código real.

**Configuração** (decisões fechadas em `docs/ESTRATEGIA.md` §1 e §3):

| item | valor |
|---|---|
| modelo base | `meta-llama/Llama-3.2-3B-Instruct` (fallback: espelho não-gated da Unsloth) |
| quantização | 4-bit NF4 + double quant, compute `float16` |
| LoRA | `r=16`, `alpha=32`, `dropout=0.05`, alvo `q_proj`/`v_proj` |
| treino | batch 4 × grad_accum 4, 3 epochs, `lr=2e-4` |

**Antes de rodar**, cadastre em `🔑 Secrets` (ícone de chave na barra lateral):
- `HF_TOKEN` — token da Hugging Face **com permissão de write** (necessário para o push do adapter no #4)
- `GOOGLE_API_KEY` **ou** `GROQ_API_KEY` — só se for gerar o dataset aqui (passo 2b)

## 0. Rodar a aplicação

**Comece por aqui.** Esta seção sobe o Portal Clínico com o modelo já treinado e
devolve o endereço de acesso. É o único passo necessário para *demonstrar* o
projeto — o fine-tuning já foi executado e o adapter está publicado no Hugging
Face Hub.

As seções 1 a 7 abaixo são o pipeline de **treino**, e só precisam rodar de novo
se o modelo for retreinado.

Pré-requisito: `Ambiente de execução > Alterar o tipo de ambiente > T4 GPU`.


In [ ]:
# Monta o ambiente e sobe a aplicação nesta máquina virtual.
#
# O script clona a `main`, instala as dependências, semeia o banco de pacientes,
# indexa o RAG e inicia o servidor. É idempotente: rodar de novo sobre uma
# sessão viva apenas atualiza o código e reinicia.
#
# Rode esta célula também quando a sessão do Colab cair — a máquina virtual é
# reciclada por inatividade e leva o ambiente inteiro junto, sem aviso.
!curl -sL https://raw.githubusercontent.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge/main/scripts/colab_portal.sh | bash

# Imprime o endereço de acesso. `localhost` não serve: aquele "local" é o da
# máquina virtual do Google, não o do seu navegador. Este endereço é a ponte
# entre os dois, e muda a cada VM nova — por isso é reimpresso aqui em vez de
# anotado em algum lugar.
from google.colab.output import eval_js
print(eval_js('google.colab.kernel.proxyPort(8501)'))


## 1. Setup

In [ ]:
# Confirma que a GPU está ativa antes de instalar 3GB de dependências.
!nvidia-smi

In [ ]:
# Dependências de treino que não estão no ambiente padrão do projeto
# (extra `finetuning` do pyproject.toml — ver ESTRATEGIA.md §10).
!pip install -q -U transformers peft accelerate datasets huggingface_hub
!pip install -q -U bitsandbytes trl
!pip install -q python-dotenv

In [ ]:
# Clona o projeto e instala o pacote em modo editável, para que
# `import hospital_assistant` resolva o código real de src/.
import os

REPO = "https://github.com/fiap-postech-ia-para-devs-grupo/9IADT-fase-3-tech-challenge.git"
BRANCH = "main"  # troque para a branch do #3 enquanto o PR não foi mergeado

if not os.path.exists("9IADT-fase-3-tech-challenge"):
    !git clone --branch $BRANCH $REPO

%cd 9IADT-fase-3-tech-challenge
!pip install -q -e . --no-deps

import sys
sys.path.insert(0, "src")

In [ ]:
# Credenciais a partir dos Secrets do Colab.
from google.colab import userdata

for chave in ("HF_TOKEN", "GOOGLE_API_KEY", "GROQ_API_KEY"):
    try:
        os.environ[chave] = userdata.get(chave)
        print(f"{chave}: carregado")
    except Exception:
        print(f"{chave}: ausente (ok se não for usar)")

In [ ]:
# Checkpoints no Drive: a sessão do Colab cai no meio do treino
# (ESTRATEGIA.md §13, risco de probabilidade ALTA). Sem isso, recomeça do zero.
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/tech-challenge-fase3/adapter"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints em:", CHECKPOINT_DIR)

## 2. Preparação de dados (#2)

`data/processed/` está no `.gitignore` (é artefato derivado), então o clone vem sem os
splits. Duas opções — **a** é a recomendada por ser reprodutível.

### 2a. Regenerar o dataset aqui

Baixa PubMedQA + MedQuAD, reaproveita o corpus sintético versionado em
`data/raw/sinteticos_finetuning.jsonl` (não gasta cota de API), anonimiza, cura,
deduplica e escreve `data/processed/{train,val}.jsonl`.

In [ ]:
# Gera o dataset sintético do hospital fictício e monta os splits de
# treino e validação. Só é necessário se o corpus for regerado.
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

from hospital_assistant.finetuning.data_prep import prepare_dataset

train_ex, val_ex = prepare_dataset()
print(f"{len(train_ex)} treino / {len(val_ex)} validação")
train_ex[0]

### 2b. (Alternativa) Subir os splits gerados na máquina local

Use se preferir treinar exatamente sobre o arquivo já revisado localmente.

In [ ]:
# from google.colab import files
# os.makedirs("data/processed", exist_ok=True)
# uploaded = files.upload()  # selecione train.jsonl e val.jsonl
# for nome in uploaded:
#     os.rename(nome, f"data/processed/{nome}")

In [ ]:
# Conferência rápida da anonimização antes de gastar GPU: nenhum exemplo
# deve conter CPF, e-mail ou telefone reais.
import json, re

with open("data/processed/train.jsonl", encoding="utf-8") as f:
    exemplos = [json.loads(l) for l in f]

suspeitos = [
    e for e in exemplos
    if re.search(r"\d{3}\.\d{3}\.\d{3}-\d{2}|[\w.]+@[\w.]+\.\w+", json.dumps(e, ensure_ascii=False))
]
print(f"{len(exemplos)} exemplos | {len(suspeitos)} com PII residual")
assert not suspeitos, suspeitos[:3]

## 3. Fine-tuning QLoRA (#3)

Toda a configuração está em `train.py` (`LORA_KWARGS`, `TRAINING_KWARGS`) e é coberta
por testes de regressão em `tests/test_train.py` — não altere os valores aqui, altere lá.

Tempo esperado no T4: **~40-70 min** para ~900 exemplos × 3 epochs.

In [ ]:
# Confere que os splits existem antes de reservar a GPU: descobrir que o
# dataset falta depois de meia hora de treino custa a sessão inteira.
from pathlib import Path
from hospital_assistant.finetuning.train import train, LORA_KWARGS, TRAINING_KWARGS

print("LoRA:  ", LORA_KWARGS)
print("Treino:", TRAINING_KWARGS)

In [ ]:
# `resume_from_checkpoint=True` retoma do último checkpoint no Drive
# se a sessão tiver caído numa tentativa anterior.
metricas = train(output_dir=Path(CHECKPOINT_DIR), resume_from_checkpoint=False)
metricas["final_eval_perplexity"]

## 4. Curvas de loss → `results/finetuning_metrics.json`

In [ ]:
# Curvas de loss a partir das métricas gravadas pelo treino. É o gráfico
# que entra no relatório técnico como evidência de que o modelo aprendeu.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([p["epoch"] for p in metricas["train"]], [p["loss"] for p in metricas["train"]], label="treino")
if metricas["eval"]:
    ax.plot([p["epoch"] for p in metricas["eval"]], [p["loss"] for p in metricas["eval"]],
            marker="o", label="validação")
ax.set_xlabel("época"); ax.set_ylabel("loss"); ax.legend()
ax.set_title(f"QLoRA — perplexidade final: {metricas['final_eval_perplexity']:.2f}")
fig.savefig("results/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Publicação do adapter no Hugging Face Hub (#4)

Sobe **só o adapter LoRA** (alguns MB), nunca os pesos do modelo base — decisão de
ESTRATEGIA.md §1 e exigência do §12 ("pesos de modelo não commitados").

In [ ]:
# Confirma qual conta o token do Colab representa antes de publicar — o
# adapter vai para o namespace dessa conta.
from huggingface_hub import HfApi, whoami

usuario = whoami(token=os.environ["HF_TOKEN"])["name"]
ADAPTER_REPO = f"{usuario}/hospital-assistant-llama32-3b-lora"
print("Publicando em:", ADAPTER_REPO)

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(ADAPTER_REPO, repo_type="model", exist_ok=True, private=False)

# `allow_patterns` é essencial: CHECKPOINT_DIR também contém os
# `checkpoint-*/` do Trainer (estado do otimizador, scheduler, RNG — centenas
# de MB). Sem o filtro, o repositório público receberia tudo isso junto, o que
# contraria a decisão de publicar "só o adapter LoRA" (ESTRATEGIA.md §1).
api.upload_folder(
    folder_path=CHECKPOINT_DIR,
    repo_id=ADAPTER_REPO,
    repo_type="model",
    allow_patterns=[
        "adapter_config.json",
        "adapter_model.safetensors",
        "tokenizer*",
        "special_tokens_map.json",
        "chat_template.jinja",
        "README.md",
    ],
)
print(f"https://huggingface.co/{ADAPTER_REPO}")

# Conferência: o repositório deve conter apenas os arquivos do adapter.
print("
Arquivos publicados:")
for arquivo in api.list_repo_files(ADAPTER_REPO):
    print(" -", arquivo)

In [ ]:
# Guarde este valor: é o que o app lê para carregar o adapter em runtime.
# Coloque no .env do projeto:  HF_ADAPTER_REPO=<valor impresso abaixo>
print(f"HF_ADAPTER_REPO={ADAPTER_REPO}")

## 6. Avaliação base vs. fine-tuned (#4)

Roda as 9 perguntas clínicas de `evaluate.PERGUNTAS_AVALIACAO` nos dois modelos e grava
`results/eval_comparativo.json`. É o insumo da seção 3.3 do relatório técnico.

In [ ]:
# Avaliação comparativa: as mesmas perguntas no modelo base e no ajustado,
# lado a lado. É o que revelou a regressão de segurança documentada no
# relatório — o ajustado passou a dar posologia onde o base recusava.
os.environ["HF_ADAPTER_REPO"] = ADAPTER_REPO

from hospital_assistant.finetuning.evaluate import evaluate, resumir

linhas = evaluate()
resumir(linhas)

In [ ]:
# Amostra das respostas para leitura humana: número isolado não mostra que
# tipo de conteúdo mudou entre os dois modelos.
for linha in linhas:
    print("=" * 100)
    print("PERGUNTA:  ", linha["question"])
    print("-" * 100)
    print("BASE:      ", linha["base_answer"][:600])
    print("-" * 100)
    print("FINE-TUNED:", linha["finetuned_answer"][:600])

## 7. Baixar os artefatos para commitar no repositório

`results/finetuning_metrics.json` e `results/eval_comparativo.json` são entregáveis
dos issues #3 e #4 e precisam entrar no Git (o adapter, não — ele vive no Hub).

In [ ]:
# Baixa os artefatos (métricas e comparativo) para commitar no repositório.
# Eles são a evidência versionada do treino; o adapter fica no Hub.
from google.colab import files

for artefato in ("results/finetuning_metrics.json", "results/eval_comparativo.json", "results/loss_curve.png"):
    if os.path.exists(artefato):
        files.download(artefato)